# Gold: hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from silver lab_results, hospitals and calendar, per the STTM `sttm_silver_to_gold.xlsx`. Includes abnormal/critical rates and a day-over-day trend via LAG. The STTM's `calendar.date_key` maps to the actual INT column `calendar.datekey`. LAG compares against the hospital's previous day that had tests, as specified by the STTM.

In [0]:
# Catalog placeholder from the STTM (<catalog>); override via widget or job parameter
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
print("Using catalog:", dbutils.widgets.get("catalog"))

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.gold.hospital_daily_summary')
COMMENT 'Gold: one row per hospital per day with lab test volume, abnormal/critical rates and day-over-day trend (STTM silver to gold v1.0)'
AS
WITH hospitals AS (
  -- dedupe defensively: one row per hospital_id
  SELECT hospital_id, hospital_name
  FROM IDENTIFIER(:catalog || '.silver.hospitals')
  QUALIFY row_number() OVER (PARTITION BY hospital_id ORDER BY update_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                                AS summary_date,
    COUNT(*)                                 AS total_tests,
    COUNT(DISTINCT patient_id)               AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM IDENTIFIER(:catalog || '.silver.lab_results')
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                             AS hospital_id,
  h.hospital_name                                                        AS hospital_name,
  w.summary_date                                                         AS summary_date,
  c.day_name                                                             AS day_name,
  c.is_weekend                                                           AS is_weekend,
  CAST(w.total_tests AS INT)                                             AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                  AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                     AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                     AS critical_test_count,
  CAST(round(w.abnormal_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(round(w.critical_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                    AS prev_day_total_tests,
  CAST(round((w.total_tests - w.prev_day_total_tests) * 100.0 / nullif(w.prev_day_total_tests, 0), 2) AS DECIMAL(6,2)) AS test_volume_change_pct
FROM with_lag w
LEFT JOIN hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN IDENTIFIER(:catalog || '.silver.calendar') c
  ON c.datekey = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)

In [0]:
import json

# Count the number of rows processed in this batch (rows written to the gold table)
target_table = f"{dbutils.widgets.get('catalog')}.gold.hospital_daily_summary"
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table}: {record_count} rows")

# End the notebook and return the record count to the calling job or notebook
dbutils.notebook.exit(json.dumps({"record_count": record_count}))